# Synthetic Data Generator Validation

**Purpose:** Before scaling up generation, test each generator prompt on a small corpus sample and compare the output against real benchmark examples.

For each generator, this notebook shows:
1. **Benchmark reference** — What a real eval example looks like (RACE, BoolQ, GSM8K, etc.)
2. **Our generator prompt** — The prompt template from `03_SYNTHETIC_DATA_GENERATORS.md`
3. **API output** — What GPT-4o-mini actually produces from a corpus sample
4. **Comparison** — Does our output match the target format and quality?

Covers generators: A (Factual), B (Chain-of-Thought), C (Reading Comprehension), D (Temporal), E (Quantitative), F (Sentence Completion), G (Instruction Following), H (Anti-Hallucination)

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Project paths
PROJECT_ROOT = Path.cwd().parent.parent  # hist_LLM/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from openai import OpenAI

# Load API key
api_key = (PROJECT_ROOT / "key.txt").read_text().strip()
client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"
PERIOD_END_YEAR = 1999  # Using 1950-1999 as test period

def call_api(prompt, temperature=0.7, max_tokens=4096):
    """Call OpenAI API and return parsed JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return json.loads(response.choices[0].message.content)

def show_json(data, title="Output"):
    """Pretty-print JSON output."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

print(f"Model: {MODEL}")
print(f"Test period end year: {PERIOD_END_YEAR}")
print("Ready.")

Model: gpt-4o-mini
Test period end year: 1999
Ready.


## Load Corpus Sample

Load a small sample of documents from the corpus. We use 3 diverse documents to test generators.

In [2]:
import pandas as pd

DATA_ROOT = Path("D:/hist_LLM")

# Try loading from actual corpus; fall back to hardcoded samples
sample_texts = {}

try:
    # Load a few documents from different sources
    synth_input = DATA_ROOT / "periods" / "1950_1999" / "posttraining_data" / "synthetic" / "input"
    
    for source in ["economist", "nyt_filtered", "ft"]:
        pq = synth_input / f"{source}.parquet"
        if pq.exists():
            df = pd.read_parquet(pq)
            # Pick a doc with decent length
            good = df[df["text"].str.len() > 2000].head(3)
            if len(good) > 0:
                row = good.iloc[0]
                sample_texts[source] = row["text"][:6000]
                print(f"Loaded {source}: {len(row['text']):,} chars (truncated to 6000)")
    
    if not sample_texts:
        raise FileNotFoundError("No parquet files found")
        
except Exception as e:
    print(f"Could not load from D: drive ({e}). Using hardcoded samples.")
    # Fallback: paste a sample document here for testing
    sample_texts["economist"] = """PASTE A SAMPLE ECONOMIST ARTICLE HERE FOR TESTING.
    
Replace this placeholder with an actual document from the corpus.
Ideal: 2000-6000 chars, containing dates, numbers, and historical events."""

print(f"\nLoaded {len(sample_texts)} sample document(s): {list(sample_texts.keys())}")

Loaded economist: 4,109 chars (truncated to 6000)
Loaded nyt_filtered: 3,036 chars (truncated to 6000)
Loaded ft: 7,932 chars (truncated to 6000)

Loaded 3 sample document(s): ['economist', 'nyt_filtered', 'ft']


---
## Generator A: Factual QA

### Benchmark Reference: MMLU / ARC-Challenge

**MMLU** (nanochat format via `render_mc()`):
```
Multiple Choice question: What is the longest river in Africa?
- Nile=A
- Congo=B
- Niger=C
- Zambezi=D

Respond only with the letter of the correct answer.
```
Model outputs: `A`

**ARC-Challenge** (same MC format):
```
Multiple Choice question: Which property of a mineral can be determined just by looking at it?
- luster=A
- mass=B
- weight=C
- hardness=D

Respond only with the letter of the correct answer.
```

**Our Generator A** currently produces open-ended QA. Below we test both the existing open-ended format AND a new MC variant.

In [3]:
# --- Generator A: Existing open-ended format ---
text = list(sample_texts.values())[0]

QA_PROMPT = """Create 3 question-answer pairs from this text for LLM training.

Rules:
1. Questions must require analytical thinking, not just fact lookup
2. Answers must be directly supported by the text
3. Vary question types: cause-effect, comparison, analysis, inference, summary
4. Return a JSON object with key "qa_pairs" containing an array:

{{"qa_pairs": [{{"question": "Question 1?", "answer": "Answer 1."}}, {{"question": "Question 2?", "answer": "Answer 2."}}]}}

Text:
{text}"""

result_a = call_api(QA_PROMPT.format(text=text))
show_json(result_a, "Generator A: Factual QA (open-ended)")


  Generator A: Factual QA (open-ended)
{
  "qa_pairs": [
    {
      "question": "What factors might contribute to the differing salary ranges for the Senior Research Officer and Research Officer positions?",
      "answer": "The salary ranges for the Senior Research Officer (SRO) and Research Officer (RO) positions are influenced by qualifications, age, and experience, as the SRO range is £7,430-28,900 while the RO range is lower at £4,980-£6,730."
    },
    {
      "question": "How does the role of the Business Development Officer at MERCEDO differ from the Research Officers in the Ministry of Defence?",
      "answer": "The Business Development Officer at MERCEDO focuses on economic studies related to the development of small firms and high technology companies, while the Research Officers in the Ministry of Defence are concerned with evaluating and assessing economic and scientific information about foreign countries."
    },
    {
      "question": "What implications does the ca

In [4]:
# --- Generator A: MC variant (to match MMLU/ARC format) ---

QA_MC_PROMPT = """Create 3 multiple-choice questions from this text for LLM evaluation.

Rules:
1. Each question must have exactly 4 options (A, B, C, D)
2. Exactly one option must be correct
3. Wrong options should be plausible but clearly incorrect
4. Questions should require analytical thinking, not just fact lookup

Return a JSON object:
{{"mc_questions": [
  {{"question": "...", "options": ["option A", "option B", "option C", "option D"], "correct": 0, "explanation": "..."}}
]}}

Text:
{text}"""

result_a_mc = call_api(QA_MC_PROMPT.format(text=text))
show_json(result_a_mc, "Generator A: Factual QA (MC variant)")

# Show how it would render in nanochat format
print("\n--- nanochat rendering ---")
for q in result_a_mc.get("mc_questions", [])[:1]:
    letters = ["A", "B", "C", "D"]
    rendered = f"Multiple Choice question: {q['question']}\n"
    for letter, opt in zip(letters, q['options']):
        rendered += f"- {opt}={letter}\n"
    rendered += "\nRespond only with the letter of the correct answer."
    print(rendered)
    print(f"\nCorrect: {letters[q['correct']]}")


  Generator A: Factual QA (MC variant)
{
  "mc_questions": [
    {
      "question": "What is the primary focus of the Business Development Officer position at MERCEDO?",
      "options": [
        "A) Conducting theoretical research in economics",
        "B) Undertaking economic studies related to small firm development",
        "C) Teaching mainstream economic theory courses",
        "D) Managing the statistical analysis of cereal production"
      ],
      "correct": 1,
      "explanation": "The Business Development Officer at MERCEDO is specifically tasked with undertaking economic studies relating to the development of small firms, particularly in high technology."
    },
    {
      "question": "Why might the Home Grown Cereals Authority prefer applicants with knowledge of computer storage of data?",
      "options": [
        "A) To ensure candidates can perform theoretical statistical analysis",
        "B) Because the position involves routine reporting and no data analysi

---
## Generator B: Chain-of-Thought Reasoning

### Benchmark Reference: HotpotQA / ARC-Challenge

**HotpotQA** (multi-hop reasoning):
```
Q: Were Scott Derrickson and Ed Wood of the same nationality?
A: Scott Derrickson is American. Ed Wood was also American. So yes, they share the same nationality.
```

**ARC-Challenge** (science reasoning requiring multi-step logic):
```
Q: Which property of a mineral can be determined just by looking at it?
A: Luster — because it is the visual appearance of light reflecting off the surface.
```

**Our Generator B** produces chain-of-thought reasoning with explicit `<think>` tags, trained from the COT_PROMPT in `run_direct.py`.

In [ ]:
# --- Generator B: Chain-of-Thought Reasoning ---
text = list(sample_texts.values())[0]

COT_PROMPT = """Create 3 complex reasoning examples from this text that
demonstrate chain-of-thought thinking.

Each example should have:
1. A challenging question that requires step-by-step reasoning
2. Detailed reasoning steps that break down the problem
3. A concise final answer

Return a JSON object with key "cot_examples" containing an array:

{{"cot_examples": [{{"question": "Complex question?",
  "reasoning": "Step 1: First, I need to consider...\\nStep 2: Then, I analyze...\\nStep 3: Finally, I can conclude...",
  "answer": "Final answer based on the reasoning."}}]}}

Text:
{text}"""

result_b = call_api(COT_PROMPT.format(text=text))
show_json(result_b, "Generator B: Chain-of-Thought Reasoning")

# Show how it renders with <think> tags
print("\n--- nanochat rendering with <think> tags ---")
for ex in result_b.get("cot_examples", [])[:2]:
    print(f"\nUser: {ex['question']}")
    print(f"Asst: <think>\n{ex['reasoning']}\n</think>\n{ex['answer']}")

---
## Generator C: Reading Comprehension

### Benchmark Reference: RACE

**RACE** (nanochat format):
```
Multiple Choice question: Read the following passage and answer the question.

Passage: The oil embargo imposed by OAPEC in 1973 had far-reaching consequences
for Western economies. Crude oil prices quadrupled from $3 to $12 per barrel
within months, triggering stagflation — simultaneous high inflation and economic
stagnation — across Europe and North America...

What was the primary economic consequence of the 1973 oil embargo according to the passage?
- Stagflation across Western economies=A
- Collapse of the OPEC cartel=B
- Rapid industrialization of oil-producing nations=C
- Immediate shift to renewable energy sources=D

Respond only with the letter of the correct answer.
```

### Benchmark Reference: BoolQ

**BoolQ** (nanochat format):
```
Read the following passage and answer True or False.

Passage: The European Economic Community was established by the Treaty of Rome
in 1957. Initially comprising six member states — Belgium, France, Germany, Italy,
Luxembourg, and the Netherlands — it aimed to create a common market...

Question: Was the EEC established before 1960?
```
Model outputs: `True`

**Key difference from Generator A:** The passage is INCLUDED in the training example.

In [13]:
# --- Generator C: Reading Comprehension ---

RC_PROMPT = """Read the following passage carefully and create 3 reading comprehension
questions with answers.

Requirements:
1. Questions should require understanding the passage, not just keyword matching
2. Include a mix of question types:
   - Extractive: "According to the passage, what/who/when..."
   - Numerical: "How many..." / "What percentage..."
   - Inferential: "What can be inferred about..."
   - Comparative: "How does X compare to Y in the passage?"
3. Answers must be directly supported by the passage text
4. For each question, also create 3 wrong options for a multiple-choice version

Return a JSON object:
{{"rc_pairs": [
  {{"question": "...",
    "answer": "...",
    "type": "extractive|numerical|inferential|comparative",
    "mc_options": ["correct answer", "wrong 1", "wrong 2", "wrong 3"]}}
]}}

Passage:
{text}"""

result_c = call_api(RC_PROMPT.format(text=text))
show_json(result_c, "Generator C: Reading Comprehension")

# Show RACE-style rendering
print("\n--- RACE-style nanochat rendering ---")
for q in result_c.get("rc_pairs", [])[:1]:
    passage_preview = text[:500] + "..."
    rendered = f"Multiple Choice question: Read the following passage and answer the question.\n\n"
    rendered += f"Passage: {passage_preview}\n\n{q['question']}\n"
    letters = ["A", "B", "C", "D"]
    for letter, opt in zip(letters, q.get('mc_options', [])):
        rendered += f"- {opt}={letter}\n"
    rendered += "\nRespond only with the letter of the correct answer."
    print(rendered)


  Generator C: Reading Comprehension
{
  "rc_pairs": [
    {
      "question": "According to the passage, what is the salary range for the Lecturer in Economics position?",
      "answer": "A$16,291 - A$21,401 per annum.",
      "type": "extractive",
      "mc_options": [
        "A$16,291 - A$21,401 per annum.",
        "A$4,980 - A$6,730 per annum.",
        "£7,430 - £28,900 per annum.",
        "£5,030 - £6,851 per annum."
      ]
    },
    {
      "question": "How many Research Officers are being recruited by the Ministry of Defence?",
      "answer": "4 Research Officers.",
      "type": "numerical",
      "mc_options": [
        "4 Research Officers.",
        "1 Research Officer.",
        "2 Research Officers.",
        "3 Research Officers."
      ]
    },
    {
      "question": "What can be inferred about the qualifications required for the Senior Research Officer position?",
      "answer": "Candidates are expected to have at least a degree with 1st or 2nd class honours 

---
## Generator F: Sentence Completion

### Benchmark Reference: HellaSwag

**HellaSwag** (nanochat format):
```
Multiple Choice question: A woman is outside with a bucket and a dog. The dog is
running around trying to avoid a bath. She...
- chases the dog and catches it, then begins to bathe it=A
- looks at the dog and walks away=B
- starts throwing the bucket at the dog=C
- picks up the bucket and carries it inside=D

Respond only with the letter of the correct answer.
```

**Our Generator F** truncates a real historical passage at a sentence boundary, then generates 4 possible continuations (1 real, 3 plausible-but-wrong). The correct answer is the actual text continuation — no GPT generation needed for ground truth.

In [ ]:
# --- Generator F: Sentence Completion ---
import re

text = list(sample_texts.values())[0]

# Step 1: Truncate at a sentence boundary (~60% through)
sentences = re.split(r'(?<=[.!?])\s+', text)
cutoff = int(len(sentences) * 0.6)
prefix = ' '.join(sentences[:cutoff])
actual_continuation = ' '.join(sentences[cutoff:cutoff+2])  # Next 1-2 sentences

COMPLETION_PROMPT = """The following is the beginning of a passage from a historical
document published in the period 1950-1999.

Passage: "{prefix}"

The actual continuation is: "{actual}"

Create 3 plausible but INCORRECT continuations that:
- Match the style and tone of the original
- Could fool someone unfamiliar with the text
- Are factually wrong, contain anachronisms, or go in the wrong direction

Return JSON:
{{"correct_continuation": "{actual}",
  "wrong_continuations": ["wrong 1", "wrong 2", "wrong 3"],
  "explanation": "Why the correct continuation is right and others are wrong"}}"""

result_f = call_api(COMPLETION_PROMPT.format(
    prefix=prefix[:2000],
    actual=actual_continuation[:500]
))
show_json(result_f, "Generator F: Sentence Completion")

# Show HellaSwag-style rendering
print("\n--- HellaSwag-style nanochat rendering ---")
import random
options = [result_f.get("correct_continuation", actual_continuation)] + \
          result_f.get("wrong_continuations", [])[:3]
correct_idx = 0
# Shuffle for display (in production, shuffle + track correct index)
print(f"Passage: {prefix[:300]}...\n")
letters = ["A", "B", "C", "D"]
for i, (letter, opt) in enumerate(zip(letters, options)):
    marker = " <-- correct" if i == correct_idx else ""
    print(f"- {opt[:150]}={letter}{marker}")

---
## Generator G: Instruction Following

### Benchmark Reference: SmolTalk / Alpaca

**SmolTalk** (multi-turn instruction):
```
User: Summarize the key points of the Treaty of Versailles.
Asst: The Treaty of Versailles (1919) had several key provisions:
      1. Germany accepted responsibility for WWI (Article 231)
      2. Significant territorial losses...
      3. Military restrictions...
      4. Reparations payments...
```

**Alpaca** (single-turn instruction):
```
Instruction: Write a brief explanation of how photosynthesis works.
Response: Photosynthesis is the process by which plants convert sunlight,
          water, and carbon dioxide into glucose and oxygen...
```

**Our Generator G** creates passage-grounded instruction-response pairs: the instruction asks for analysis, summarization, or explanation, and the response must be supported by the source document.

In [ ]:
# --- Generator G: Instruction Following ---
text = list(sample_texts.values())[0]

INSTRUCTION_PROMPT = """From the following historical document, create 3 instruction-response
pairs suitable for training an instruction-following LLM.

Requirements:
1. Instructions should be diverse: include summarization, analysis, comparison,
   explanation, and "based on the passage" tasks
2. Responses must be grounded in the passage (no hallucinated facts)
3. Responses should be detailed (2-4 sentences minimum)
4. Vary the instruction style:
   - Direct: "Summarize the main points of..."
   - Contextual: "Based on the passage, explain why..."
   - Analytical: "What are the implications of..."

Return JSON:
{{"instruction_pairs": [
  {{"instruction": "...",
    "response": "...",
    "type": "summarization|analysis|comparison|explanation"}}
]}}

Passage:
{text}"""

result_g = call_api(INSTRUCTION_PROMPT.format(text=text))
show_json(result_g, "Generator G: Instruction Following")

# Show instruction-response rendering
print("\n--- Training example renderings ---")
for pair in result_g.get("instruction_pairs", []):
    print(f"\n[{pair.get('type', 'unknown').upper()}]")
    print(f"User: {pair['instruction'][:200]}")
    print(f"Asst: {pair['response'][:300]}")

---
## Generator D: Temporal Reasoning

### Benchmark Reference: TimE (arXiv:2505.12891)

**Level 1 — Date extraction:**
```
Q: When was the North Atlantic Treaty signed?
A: April 4, 1949
```

**Level 2 — Relative reasoning:**
```
Q: What major international organization was established in the decade
   before the Korean War began?
A: The United Nations (1945), approximately 5 years before the Korean War (1950)
```

**Level 3 — Timeline ordering:**
```
Q: Arrange in chronological order: Marshall Plan, NATO formation,
   Berlin Blockade, Korean War
A: Marshall Plan (1948) -> Berlin Blockade (1948-49) -> NATO (1949) -> Korean War (1950)
```

**No existing dataset covers all 3 levels from a historical corpus. This is unique to our project.**

In [6]:
# --- Generator D Level 1: Basic Temporal (single document) ---

TEMPORAL_L1_PROMPT = """From the following historical document, create 3 questions
that test basic temporal understanding.

For each question, also specify the FORMAT from: mc, cot, true_false, ranking

Required question subtypes (include at least one of each):
- Date/time extraction: "When did [event] happen?" -> format: mc or true_false
- Duration computation: "How long did [period/event] last?" -> format: cot
- Simple ordering: "Did [A] happen before or after [B]?" -> format: true_false or mc

Each question MUST have a definitive answer derivable from the text.

Return JSON:
{{"temporal_qa": [
  {{"question": "...",
    "answer": "...",
    "format": "mc|cot|true_false",
    "level": 1,
    "subtype": "extraction|duration|ordering",
    "mc_options": ["A", "B", "C", "D"]
  }}
]}}

Document:
{text}"""

result_d1 = call_api(TEMPORAL_L1_PROMPT.format(text=text))
show_json(result_d1, "Generator D Level 1: Basic Temporal")

# Show different format renderings
print("\n--- Format rendering examples ---")
for q in result_d1.get("temporal_qa", []):
    fmt = q.get("format", "open_ended")
    print(f"\n[{fmt.upper()}] {q.get('subtype', 'unknown')}")
    if fmt == "mc" and q.get("mc_options"):
        print(f"Q: {q['question']}")
        for letter, opt in zip('ABCD', q['mc_options']):
            print(f"  {letter}) {opt}")
    elif fmt == "true_false":
        print(f"Q: {q['question']}")
        print(f"A: {q['answer']}")
    elif fmt == "cot":
        print(f"Q: {q['question']}")
        print(f"A: {q['answer']}")


  Generator D Level 1: Basic Temporal
{
  "temporal_qa": [
    {
      "question": "When did the appointment for the Lecturer in Economics take place?",
      "answer": "29 February 1980",
      "format": "mc",
      "level": 1,
      "subtype": "extraction",
      "mc_options": [
        "29 February 1980",
        "1 March 1980",
        "28 February 1980",
        "29 January 1980"
      ]
    },
    {
      "question": "How long did the appointment for the Lecturer in Economics last?",
      "answer": "2 years",
      "format": "cot",
      "level": 1,
      "subtype": "duration"
    },
    {
      "question": "Did the application deadline for the Civil Service Commission happen before or after the appointment date for the Lecturer in Economics?",
      "answer": "after",
      "format": "true_false",
      "level": 1,
      "subtype": "ordering"
    }
  ]
}

--- Format rendering examples ---

[MC] extraction
Q: When did the appointment for the Lecturer in Economics take place?
  

In [7]:
# --- Generator D Level 2: Cross-Document (uses 2 source documents) ---
# This requires 2 documents. If we have multiple sources, use them.

sources = list(sample_texts.keys())
if len(sources) >= 2:
    text1 = sample_texts[sources[0]]
    text2 = sample_texts[sources[1]]
    source1, source2 = sources[0], sources[1]
else:
    # Split single document in half as fallback
    full = list(sample_texts.values())[0]
    mid = len(full) // 2
    text1, text2 = full[:mid], full[mid:]
    source1, source2 = "document_part1", "document_part2"

TEMPORAL_L2_PROMPT = """Given these two passages from the same historical period,
create 2 questions that require connecting information across them.

For each question, specify FORMAT from: cot, ranking

Required question subtypes:
- Contemporaneous: "What was happening in [domain B] while [event A] occurred?"
- Causal: "How might [earlier event in Passage 1] have influenced [later event in Passage 2]?"
- Period characterization: "Based on both passages, what defined the [decade]?"

Passage 1 ({source1}):
{text1}

Passage 2 ({source2}):
{text2}

Return JSON:
{{"temporal_qa": [
  {{"question": "...",
    "answer": "...",
    "format": "cot|ranking",
    "level": 2,
    "subtype": "contemporaneous|causal|characterization",
    "reasoning": "Step-by-step reasoning connecting the two passages..."
  }}
]}}"""

result_d2 = call_api(TEMPORAL_L2_PROMPT.format(
    source1=source1, text1=text1[:3000],
    source2=source2, text2=text2[:3000]
))
show_json(result_d2, "Generator D Level 2: Cross-Document Temporal")


  Generator D Level 2: Cross-Document Temporal
{
  "temporal_qa": [
    {
      "question": "What was happening in the field of economics and education in the UK while the maintenance of the Williamsburg Bridge in New York City was halted due to lead paint concerns?",
      "answer": "In the UK, there were multiple job appointments and research opportunities in economics and education, including positions at the University of Queensland and the Ministry of Defence, indicating a focus on economic research and educational planning during the same period.",
      "format": "cot",
      "level": 2,
      "subtype": "contemporaneous",
      "reasoning": "This question connects the economic and educational developments in the UK with the public health and safety issues arising in New York, illustrating the broader socio-economic context of the time."
    },
    {
      "question": "How might the economic surplus reported in Hartford have influenced the protests against layoffs in New Jersey

---
## Generator E: Corpus-Grounded Quantitative Reasoning

### Benchmark Reference: GSM8K

**GSM8K format:**
```
Q: Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning
   and bakes muffins for her friends every day with four. She sells every remaining
   egg at the farmers' market daily for $2 per egg. How much in dollars does she
   make every day at the farmers' market?

A: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
   She makes 9 * 2 = <<9*2=18>>$18 every day.
   #### 18
```

**Key difference:** GSM8K uses invented numbers. Our Generator E uses REAL numbers from historical documents (trade figures, economic data, demographics).

**Pre-filter:** Only send document chunks that contain sufficient numerical content.

In [8]:
import re

def has_sufficient_numbers(text, min_numbers=3):
    """Check if text has enough numerical content for math problems."""
    patterns = [
        r'\$[\d,.]+\s*(million|billion|thousand)?',   # currency
        r'[\d,.]+\s*%',                                # percentages
        r'[\d,.]+\s*(million|billion|thousand|tons|bushels|barrels)',  # quantities
        r'\b\d{1,3}(,\d{3})+\b',                     # large numbers
    ]
    matches = sum(len(re.findall(p, text, re.IGNORECASE)) for p in patterns)
    return matches, matches >= min_numbers

# Check which samples have enough numbers
print("Numeric content check:")
best_text = None
for source, txt in sample_texts.items():
    count, sufficient = has_sufficient_numbers(txt)
    status = "OK" if sufficient else "LOW"
    print(f"  {source}: {count} numeric patterns [{status}]")
    if sufficient and best_text is None:
        best_text = txt

if best_text is None:
    best_text = list(sample_texts.values())[0]
    print("\n  Warning: No sample has 3+ numeric patterns. Using first sample anyway.")

Numeric content check:
  economist: 11 numeric patterns [OK]
  nyt_filtered: 6 numeric patterns [OK]
  ft: 17 numeric patterns [OK]


In [9]:
# --- Generator E: Quantitative Reasoning ---

QUANTITATIVE_PROMPT = """From the following historical document containing numerical
data, create 3 math word problems that require step-by-step calculation.

Requirements:
1. Use ONLY numbers that actually appear in the text (prices, percentages,
   populations, trade volumes, etc.)
2. Each problem should require 2-4 calculation steps
3. Show complete step-by-step solutions
4. For each problem, specify FORMAT from: open_ended, cot, fill_blank

Problem types to vary across:
- Percentage change: "By what percentage did X change from Y to Z?"
- Compound growth: "If X grew at Y% annually, what was the value after N years?"
- Ratio comparison: "What was the ratio of X to Y?"
- Difference: "How much more/less was X compared to Y?"

Return JSON:
{{"math_problems": [
  {{"problem": "...",
    "solution": "Step 1: ...\\nStep 2: ...\\nStep 3: ...",
    "answer": "...",
    "format": "open_ended|cot|fill_blank",
    "source_numbers": ["$240 million", "8%"]
  }}
]}}

Historical document:
{text}"""

result_e = call_api(QUANTITATIVE_PROMPT.format(text=best_text))
show_json(result_e, "Generator E: Quantitative Reasoning")

# Show different format renderings
print("\n--- Format comparisons ---")
for p in result_e.get("math_problems", []):
    fmt = p.get("format", "open_ended")
    print(f"\n[{fmt.upper()}]")
    if fmt == "cot":
        print(f"Q: {p['problem']}")
        print(f"A: <think>\n{p['solution']}\n</think>\n{p['answer']}")
    elif fmt == "fill_blank":
        print(f"Q: {p['problem']}")
        print(f"A: {p['answer']}")
    else:
        print(f"Q: {p['problem']}")
        print(f"A: {p['solution']}\nFinal: {p['answer']}")
    print(f"Source numbers: {p.get('source_numbers', [])}")


  Generator E: Quantitative Reasoning
{
  "math_problems": [
    {
      "problem": "By what percentage did the starting salary of the Research Officer (RO) increase from £4,980 to £6,730?",
      "solution": "Step 1: Calculate the increase in salary: £6,730 - £4,980 = £1,750.\nStep 2: Calculate the percentage increase: (£1,750 / £4,980) * 100 = 35.14%.\nStep 3: Round to two decimal places, the percentage increase is approximately 35.14%.",
      "answer": "35.14%",
      "format": "open_ended",
      "source_numbers": [
        "£4,980",
        "£6,730"
      ]
    },
    {
      "problem": "If the salary of the Senior Research Officer (SRO) grew at an annual rate of 8% starting from the minimum of £7,430, what will be the salary after 2 years?",
      "solution": "Step 1: Calculate the salary after the first year: £7,430 * (1 + 0.08) = £7,430 * 1.08 = £8,015.40.\nStep 2: Calculate the salary after the second year: £8,015.40 * (1 + 0.08) = £8,015.40 * 1.08 = £8,634.00.\nStep 3: Afte

---
## Generator H: Anti-Hallucination

### Why This Generator Exists

**Problem:** LLMs hallucinate by default. If asked about something outside their training period, they fabricate answers rather than refusing.

**Academic backing:**
- **R-Tuning** (Zhang et al., NeurIPS 2023): Training with refusal examples significantly reduces hallucination
- **Gekhman et al.** (EMNLP 2024): SFT on facts not seen in pretraining *increases* hallucination — so fact memorization via SFT backfires

**Generator H teaches the model WHERE ITS KNOWLEDGE STOPS** — not what it knows (that's Generator A + pretraining).

This generator does NOT use corpus text as input. It generates questions about post-period events and trains the model to refuse appropriately.

In [10]:
# --- Generator H Subtype 1: Direct Post-Period Questions ---

ANTI_HALLUC_DIRECT_PROMPT = """Generate 5 question-answer pairs where the question
asks about events, technologies, or developments that occurred AFTER the year {end_year}.

For each pair, specify FORMAT from: mc, open_ended, true_false

The correct response must ALWAYS:
1. Acknowledge the question
2. State clearly that the information is beyond the model's knowledge period
3. Offer what WAS known as of {end_year} (if relevant context exists)
4. VARY the refusal phrasing across pairs

Topics to cover (one per pair):
- Technology invented after {end_year}
- Political event after {end_year}
- Cultural phenomenon after {end_year}
- Scientific discovery after {end_year}
- Company or organization founded after {end_year}

For MC format: one option should be "This occurred after my knowledge period."
For T/F: make a statement about a post-period fact; answer should explain inability to verify.

Return JSON:
{{"anti_halluc_pairs": [
  {{"question": "...",
    "answer": "...",
    "format": "mc|open_ended|true_false",
    "event_year": 2005,
    "domain": "technology|politics|culture|science|economics",
    "mc_options": ["opt A", "opt B", "opt C", "opt D"]
  }}
]}}"""

result_h1 = call_api(ANTI_HALLUC_DIRECT_PROMPT.format(end_year=PERIOD_END_YEAR))
show_json(result_h1, f"Generator H: Anti-Hallucination (post-{PERIOD_END_YEAR})")

# Show format examples
print("\n--- Training example renderings ---")
for pair in result_h1.get("anti_halluc_pairs", [])[:3]:
    fmt = pair.get("format", "open_ended")
    print(f"\n[{fmt.upper()}] Domain: {pair.get('domain', '?')} | Event year: {pair.get('event_year', '?')}")
    print(f"User: {pair['question'][:200]}")
    print(f"Asst: {pair['answer'][:300]}")


  Generator H: Anti-Hallucination (post-1999)
{
  "anti_halluc_pairs": [
    {
      "question": "What new technology was invented after 1999 that revolutionized mobile communication?",
      "answer": "I'm unable to provide details about technological advancements made after my knowledge cutoff in 2021. However, prior to 1999, mobile communication was primarily dominated by analog systems and the early stages of digital networks.",
      "format": "open_ended",
      "event_year": 2007,
      "domain": "technology"
    },
    {
      "question": "Can you describe a significant political event that took place after 1999?",
      "answer": "I cannot confirm details about political events that occurred after 2021. Before that time, the late 1990s saw the end of the Cold War and various international treaties being signed.",
      "format": "open_ended",
      "event_year": 2008,
      "domain": "politics"
    },
    {
      "question": "What cultural phenomenon became popular in the 200

In [11]:
# --- Generator H Subtype 2: Boundary Probing ---

BOUNDARY_PROBE_PROMPT = """Generate 3 questions that probe the boundary of knowledge
at year {end_year}.

These should be questions where:
- A partial answer exists within the period (ongoing trend, unresolved issue)
- A full/updated answer requires post-{end_year} knowledge
- The model should provide the partial answer AND explicitly note the limitation

The answer should follow this pattern:
"As of {end_year}, [what was known/happening]. [The situation was still developing /
The outcome had not yet been determined / Further developments occurred after my
knowledge period.]"

Example for end_year=1999: Asking about the outcome of the Kosovo conflict.

Return JSON:
{{"boundary_pairs": [
  {{"question": "...",
    "answer": "...",
    "format": "open_ended",
    "context_note": "What actually happened after {end_year} (not shown to model)"
  }}
]}}"""

result_h2 = call_api(BOUNDARY_PROBE_PROMPT.format(end_year=PERIOD_END_YEAR))
show_json(result_h2, f"Generator H: Boundary Probing (at {PERIOD_END_YEAR})")

# Show examples with the hidden context
print("\n--- Boundary probe examples ---")
for pair in result_h2.get("boundary_pairs", []):
    print(f"\nQ: {pair['question'][:200]}")
    print(f"A: {pair['answer'][:300]}")
    print(f"[Hidden context: {pair.get('context_note', 'N/A')[:200]}]")


  Generator H: Boundary Probing (at 1999)
{
  "boundary_pairs": [
    {
      "question": "What were the key factors contributing to the dot-com bubble, and what was the potential for its future impact on the economy?",
      "answer": "As of 1999, the dot-com bubble was characterized by excessive investment in internet-based companies, leading to rapidly rising stock prices and speculative behavior. Many believed that the internet would revolutionize commerce and communication. However, the situation was still developing, and the outcome had not yet been determined as the bubble was projected to burst in the early 2000s, which would have significant economic implications. Further developments occurred after my knowledge period.",
      "format": "open_ended",
      "context_note": "The dot-com bubble burst in 2000, leading to a significant market decline."
    },
    {
      "question": "What were the ongoing debates surrounding climate change, and how was international policy addres

---
## Summary: Generator Output vs. Benchmark Format

After running the cells above, compare:

| Generator | Target Benchmark | Key Check |
|-----------|-----------------|----------|
| A (open-ended) | General SFT | Are questions analytical? Do answers cite the text? |
| A (MC variant) | MMLU / ARC | Do options render correctly in nanochat `render_mc()` format? |
| B (CoT) | HotpotQA / ARC | Are reasoning steps explicit? Does `<think>` wrapping work? |
| C | RACE / BoolQ | Is the passage included? Are question types varied? |
| D Level 1 | TimE Level 1 | Are dates/durations/orderings extractable from source? |
| D Level 2 | TimE Level 2-3 | Do questions genuinely require BOTH passages? |
| E | GSM8K | Are source numbers real? Are solutions arithmetically correct? |
| F | HellaSwag | Is the correct continuation the actual text? Are distractors plausible? |
| G | SmolTalk / Alpaca | Are instructions diverse? Are responses passage-grounded? |
| H Direct | R-Tuning | Are refusals varied? Do they offer pre-period context? |
| H Boundary | R-Tuning | Does the model give partial answers + flag limitations? |

### Quality Checklist
- [ ] JSON parses correctly (no format errors)
- [ ] Questions are non-trivial (not just keyword lookup)
- [ ] MC options are plausible (not obviously wrong)
- [ ] CoT reasoning has clear multi-step logic
- [ ] Sentence completions have plausible distractors matching source style
- [ ] Instruction-response pairs are grounded in the passage
- [ ] Temporal questions actually require temporal reasoning
- [ ] Math problems use REAL numbers from the source text
- [ ] Refusal phrasings are diverse (not repetitive)
- [ ] Boundary probes give partial answers (not just refusals)

In [ ]:
# --- Quick stats ---
results = {
    "A (open-ended)": result_a.get("qa_pairs", []),
    "A (MC)": result_a_mc.get("mc_questions", []),
    "B (CoT)": result_b.get("cot_examples", []),
    "C (RC)": result_c.get("rc_pairs", []),
    "D L1 (temporal)": result_d1.get("temporal_qa", []),
    "D L2 (cross-doc)": result_d2.get("temporal_qa", []),
    "E (quantitative)": result_e.get("math_problems", []),
    "F (completion)": [result_f] if result_f else [],
    "G (instruction)": result_g.get("instruction_pairs", []),
    "H (direct refusal)": result_h1.get("anti_halluc_pairs", []),
    "H (boundary)": result_h2.get("boundary_pairs", []),
}

print("Generation Summary")
print("=" * 50)
total = 0
for name, items in results.items():
    n = len(items)
    total += n
    print(f"  {name}: {n} examples")
print(f"\n  Total: {total} examples generated")
print(f"  Generators covered: A, B, C, D, E, F, G, H (all 8)")
print(f"\nReview the outputs above and check the quality checklist.")
print(f"If prompts need tuning, edit them in the cells above and re-run.")